# Building a Chatbot + Guardrail with OpenAI API (No RAG)

# Architecture
```
Customer Input
   ↓
Intent Classification Using LLM  (refund_request, product_query, complaint)
   ↓
Guardrail Check (Input)
   ↓
LLM Response (JSON formatted)
   ↓
Guardrail Check (Output)
   ↓
Final Response To Customer

## 1. Setup

In [ ]:
# Install OpenAI (run once)
!pip install openai

In [2]:
# check python , openai version

import openai
print(openai.__version__)

2.29.0


In [3]:
# Check python version

!python --version

Python 3.12.2


In [7]:
from openai import OpenAI

# Initialize client
# NOT A GOOD PRACTISE TO PUT KEY HERE
API_KEY="_key_here" 

client = OpenAI(api_key=API_KEY)

## 2. Build Basic Chatbot

In [8]:
system_prompt = """
You are a polite customer support assistant for an e-commerce company.

Rules:
- Be polite and professional
- Ask for order ID when needed
- Handle refunds, complaints, and product queries
"""

messages = [
    {"role": "system", "content": system_prompt}
]

while True:
    user_input = input("You: ") # Example: I need to return shoes and want refund.
    
    if user_input.lower() == "exit":
        break
    
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )
    
    reply = response.choices[0].message.content
    print("Bot:", reply)
    
    messages.append({"role": "assistant", "content": reply}) # This way it remember previous response


You:  I need to return shoes and want refund.


Bot: I'm here to help you with that! Could you please provide me with your order ID? This will help me locate your order and assist you with the return and refund process. Thank you!


You:  exit


# EXTENSION 1: Intent Classification

## Step 1: Create Intent Classifier

In [9]:
def classify_intent(user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Classify the user intent into one of: \
                            refund_request, product_query, complaint. \
                            Return only the label."
            },
            {"role": "user", "content": user_input}
        ],
        temperature=0
    )
    
    return response.choices[0].message.content.strip()


## Step 2: Test Intent Classification


In [12]:
# user_input = "I want to return my product. I want my money back"  # refund
# user_input = "How much is the Radio? Any discount on the product ?" # product_query  
user_input = "The product came broken. It was missing handle. It had nasty scratches on the side"  # complaint

intent = classify_intent(user_input)

print("User Input:", user_input)
print("Detected Intent:", intent)

User Input: The product came broken. It was missing handle. It had nasty scratches on the side
Detected Intent: complaint


## Step 3: Use Intent in Chatbot


In [13]:
user_input = input("You: ")

intent = classify_intent(user_input)
print("Detected Intent:", intent)

if intent == "refund_request":
    print("Bot: Sure, please provide your order ID.")
elif intent == "complaint":
    print("Bot: I'm sorry for the inconvenience. Can you explain the issue?")
else:
    print("Bot: Let me help you with that.")

You:  I want to return my product. I want my money back


Detected Intent: refund_request
Bot: Sure, please provide your order ID.


# EXTENSION 2: JSON Response Formatting

##  Step 1: Define Structured Prompt

In [14]:
system_prompt = """
You are a polite customer support assistant for an e-commerce company.

Rules:
- Be polite and professional
- Ask for order ID when needed
- Handle refunds, complaints, and product queries

Always respond in JSON format:
{
  "intent": "...",
  "message": "...",
  "action_required": true/false
}
"""


## Step 2: Generate JSON Response

In [15]:
user_input = input("You: ")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ],
    temperature=0.5
)

reply = response.choices[0].message.content
print("Raw JSON Response:\n", reply)

You:  I want to return my product. I want my money back


Raw JSON Response:
 {
  "intent": "return_request",
  "message": "I can assist you with the return process. Could you please provide your order ID?",
  "action_required": true
}


## Step 3: Convert JSON String to Python Dict

In [16]:
import json

try:
    parsed = json.loads(reply)
    print("\nParsed Output:")
    print("Intent:", parsed["intent"])
    print("Message:", parsed["message"])
    print("Action Required:", parsed["action_required"])
except:
    print("Error parsing JSON")



Parsed Output:
Intent: return_request
Message: I can assist you with the return process. Could you please provide your order ID?
Action Required: True


# EXTENSION 3: Guardrails

## Step 1: Input Guardrail

In [17]:
# Prevent user from using blocked words
blocked_words = ["hack", "illegal", "fraud"]

def is_safe(user_input):
    return not any(word in user_input.lower() for word in blocked_words)


## Step 2: Test Guardrail

In [19]:
user_input = input("You: ") # I want to hack, commit fraud. Because I am illegal. How can I do this?

if not is_safe(user_input):
    print("Bot: I cannot assist with that request.")
else:
    print("Bot: Input is safe.")


You:  I want refund


Bot: Input is safe.


## Step 3: Add Guardrail to Chatbot

In [ ]:
system_prompt = """
You are a polite customer support assistant for an e-commerce company.

Rules:
- Be polite and professional
- Ask for order ID when needed
- Handle refunds, complaints, and product queries

Always respond in JSON format:
{
  "intent": "...",
  "message": "...",
  "action_required": true/false
}
"""

messages = [
    {"role": "system", "content": system_prompt}
]


while True:
    user_input = input("You: ")
    
    if user_input.lower() == "exit":
        break
    
    # Input Guardrail
    if not is_safe(user_input):
        print("Bot: I cannot assist with that request.")
        continue
    
    messages.append({"role": "user", "content": user_input})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7
    )
    
    reply = response.choices[0].message.content
    
    # Output Guardrail (simple): Here I Also Check Output from LLM
    if "illegal" in reply.lower():
        reply = "Bot: I am not allowed to provide that information."
    
    print("Bot:", reply)

    messages.append({"role": "assistant", "content": reply}) # This way it remember previous response

You:  I need refund


Bot: {
  "intent": "refund_request",
  "message": "I can assist you with your refund request. Could you please provide me with your order ID?",
  "action_required": true
}


You:  I want to hack


Bot: I cannot assist with that request.


# DELETE THE API KEY

## Chatbot Architecture:

1. User Input
2. Intent Classification
3. Guardrail Check (Input)
4. LLM Response Generation
5. JSON Formatting
6. Guardrail Check (Output)
7. Final Response to User

## Assignment:

1. Add new intent: order_status
2. Add 3 more blocked words
3. Modify chatbot tone (rude / friendly)